# Projeto final - Visualizador 3D de Moléculas
**Autora:** Maria Vitória Dias Meneses
<br>
**Professores Responsáveis:**
<br>
Prof. Dr. Daniel Roberto Cassar
<br>
Prof. Dr. James Moraes de Almeida
<br>
Prof. Dr. Leandro Nascimento Lemos
<br>
**Instituição:** Ilum Escola de Ciência / CNPEM
<br>
**Disciplina:** Práticas Básicas em Ciência de Dados


## Sobre o projeto
 
Este projeto une dois conceitos centrais da disciplina — **grafos** e **ciência de dados** — aplicados à visualização de estruturas moleculares em três dimensões.
 
A ideia central é simples: uma molécula é, matematicamente, um grafo. Cada átomo é um nó e cada ligação química é uma aresta. A partir dessa representação, é possível não só visualizar a geometria da molécula no espaço, mas também extrair propriedades estruturais diretamente da teoria dos grafos, como grau dos nós, diâmetro e conectividade.
 
O usuário fornece uma notação SMILES — padrão amplamente utilizado em química computacional — e o programa gera automaticamente a estrutura 3D interativa da molécula, junto com uma análise do grafo correspondente.

## Bibliotecas utilizadas

Para este projeto, utilizou-se cinco bibliotecas externas, cada uma com uma função específica no pipeline.

| Biblioteca | Função |
|---|---|
| **RDKit** | Interpreta o SMILES, adiciona hidrogênios e gera as coordenadas 3D dos átomos |
| **NetworkX** | Representa a molécula como grafo e calcula propriedades como grau e diâmetro |
| **Plotly** | Renderiza a visualização 3D interativa no notebook |
| **PubChemPy** | Consulta o banco de dados do PubChem para identificar o nome da molécula |
| **NumPy** | Oferece suporte a operações matemáticas auxiliares |

A última linha configura o Plotly para renderizar os gráficos diretamente nas células do JupyterLab.

In [14]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import networkx as nx
import plotly.graph_objects as go
import numpy as np
import pubchempy as pcp
import plotly.io as pio

pio.renderers.default = "notebook"

## Configurações visuais

Para tornar a visualização mais intuitiva, cada elemento químico é representado por uma cor específica, seguindo uma convenção amplamente utilizada em softwares de química molecular (como o padrão CPK). As cores são definidas em hexadecimal e podem ser personalizadas livremente no dicionário abaixo.

In [15]:
cores_atomos = {
    "C":  "#404040",   # Carbono — cinza escuro
    "H":  "#FFFFFF",   # Hidrogênio — branco
    "O":  "#FF4444",   # Oxigênio — vermelho
    "N":  "#4444FF",   # Nitrogênio — azul
    "S":  "#FFFF00",   # Enxofre — amarelo
    "P":  "#FF8800",   # Fósforo — laranja
    "F":  "#88FF88",   # Flúor — verde claro
    "Cl": "#00CC00",   # Cloro — verde
    "Br": "#884400",   # Bromo — marrom
    "I":  "#660099",   # Iodo — roxo
}

Além das cores, cada elemento possui um tamanho de esfera proporcional ao seu raio atômico relativo — átomos maiores aparecem maiores na visualização. Para elementos que não estejam listados nos dicionários, são utilizados uma cor cinza e um tamanho padrão como substituto.

In [16]:
# Raio (tamanho) de cada átomo na visualização
raio_atomos = {
    "C": 12, "H": 6,  "O": 14, "N": 13,
    "S": 16, "P": 15, "F": 10, "Cl": 16,
    "Br": 18, "I": 20,
}
 
# Cor padrão para elementos não listados acima
cor_padrao  = "#AAAAAA"
raio_padrao = 10

O PubChemPy retorna os nomes das moléculas em inglês. Para exibir os nomes em português, utilizamos um dicionário de traduções que mapeia o nome em inglês para o equivalente em português. Caso a molécula não esteja listada, o nome em inglês é mantido automaticamente.

In [17]:
traducoes = {
    "water" : "Água",
    "methane" : "Metano",
    "ethanol" : "Etanol",
    "benzene" : "Benzeno",
    "ammonia" : "Amônia",
    "glucose" : "Glicose",
    "acetic acid" : "Ácido acético",
    "cyclohexane" : "Ciclohexano",
    "naphthalene" : "Naftaleno",
    "oxygen" : "Oxigênio",
    "carbon dioxide" : "Dióxido de carbono",
    "methanol" : "Metanol",
    "acetone" : "Acetona",
    "sucrose" : "Sacarose",
    "caffeine" : "Cafeína",
    "propane" : "Propano",
    "hydrogen peroxide" : "Peróxido de hidrogênio",
    "ethane" : "Etano",
    "butane" : "Butano",
    "pentane" : "Pentano"
}

## Funções

### Smiles para nomes
A função abaixo utiliza o PubChemPy para consultar o banco de dados do PubChem e recuperar o nome comum da molécula a partir do seu SMILES. O nome retornado é então comparado ao dicionário de traduções — se houver correspondência, o nome em português é utilizado; caso contrário, o nome em inglês é mantido. Se a consulta falhar por qualquer motivo, o próprio SMILES é retornado como identificador.

In [18]:
def smiles_para_nome(smiles: str) -> str:
    try:
        compostos = pcp.get_compounds(smiles, namespace='smiles')
        if compostos:
            sinonimos = compostos[0].synonyms
            if sinonimos:
                nome_ingles = sinonimos[0]
                return traducoes.get(nome_ingles.lower(), nome_ingles)
    except:
        pass
    return smiles  # se não achar, retorna o próprio SMILES

### Construindo a molécula em 3D

A primeira etapa do pipeline é converter o SMILES em uma molécula com coordenadas tridimensionais reais. A função abaixo realiza esse processo em três passos: primeiro interpreta o SMILES e cria o objeto molecular; em seguida adiciona os hidrogênios explicitamente, já que por padrão o RDKit os omite; por fim, gera as coordenadas 3D usando um algoritmo de geometria molecular e otimiza a estrutura com o campo de força **MMFF94**, um modelo matemático que simula as forças entre os átomos para encontrar a conformação de menor energia.

Vale destacar que o parâmetro `randomSeed=42` garante que a mesma conformação seja gerada toda vez que o código for executado, tornando o resultado reprodutível.

In [19]:
def construir_molecula(smiles: str):
    
    mol = Chem.MolFromSmiles(smiles)
 
    if mol is None:
        print(f" SMILES inválido: '{smiles}'")
        print("   Verifique a notação. Exemplo: água = 'O', metano = 'C'")
        return None
    
    mol = Chem.AddHs(mol)
 
    resultado = AllChem.EmbedMolecule(mol, randomSeed=42)
 
    if resultado == -1:
        print(" Não foi possível gerar coordenadas 3D para essa molécula.")
        return None
 
    AllChem.MMFFOptimizeMolecule(mol)
 
    return mol

### Convertendo a molécula em grafo

Com a molécula construída, o próximo passo é representá-la como um grafo matemático — e é aqui que a conexão entre química e teoria dos grafos se torna concreta. A função abaixo percorre todos os átomos da molécula e os adiciona como **nós** do grafo, guardando o símbolo do elemento e as coordenadas 3D como atributos. Em seguida, percorre todas as ligações químicas e as adiciona como **arestas**, registrando o tipo de cada ligação: simples (1.0), dupla (2.0), tripla (3.0) ou aromática (1.5), como no caso do benzeno.

O resultado é um grafo NetworkX que carrega toda a informação estrutural da molécula e está pronto tanto para análise quanto para visualização.

In [20]:
def molecula_para_grafo(mol):
    
    G = nx.Graph()
 
    conformacao = mol.GetConformer()
 
    for atomo in mol.GetAtoms():
        idx      = atomo.GetIdx()
        simbolo  = atomo.GetSymbol()
        pos      = conformacao.GetAtomPosition(idx)
 
        G.add_node(
            idx,
            simbolo = simbolo,
            x       = pos.x,
            y       = pos.y,
            z       = pos.z,
        )
 
    for ligacao in mol.GetBonds():
        i    = ligacao.GetBeginAtomIdx()
        j    = ligacao.GetEndAtomIdx()
        tipo = str(ligacao.GetBondTypeAsDouble())
 
        G.add_edge(i, j, tipo=tipo)
 
    return G

### Analisando o grafo

Uma das vantagens de representar a molécula como grafo é a possibilidade de extrair propriedades estruturais usando ferramentas da teoria dos grafos. A função abaixo imprime um resumo dessas propriedades para qualquer molécula fornecida.

O **número de nós** corresponde ao total de átomos, incluindo os hidrogênios. O **número de arestas** corresponde ao total de ligações químicas. O **diâmetro do grafo** é a maior distância entre quaisquer dois átomos, medida em número de ligações — na água, por exemplo, o diâmetro é 2, pois o caminho mais longo é H→O→H. O **grau de cada nó** indica quantas ligações aquele átomo possui, o que em química corresponde à sua valência. Por fim, a **composição** lista quantos átomos de cada elemento estão presentes na molécula.

Para moléculas pequenas, a lista de graus é exibida em uma única coluna. Já para moléculas maiores — acima de 15 átomos —, a lista é dividida automaticamente em duas colunas lado a lado, tornando a leitura mais compacta e organizada.

In [21]:
def analisar_grafo(G, nome_molecula="Molécula"):
    
    print(f"\n{'='*50}")
    print(f"  Análise da molécula — {nome_molecula}")
    print(f"{'='*50}")
    print(f"  Nós (átomos)       : {G.number_of_nodes()}")
    print(f"  Arestas (ligações) : {G.number_of_edges()}")
    print(f"  Diâmetro do grafo  : {nx.diameter(G)}")
 
    print(f"\n  Grau de cada átomo (nº de ligações):")
    nos = list(G.nodes(data=True))
    limite = 15  # a partir de quantos átomos divide em duas colunas

    if len(nos) > limite:
        metade = (len(nos) + 1) // 2
        coluna1 = nos[:metade]
        coluna2 = nos[metade:]

        for i in range(metade):
            no1, dados1 = coluna1[i]
            texto1 = f"    Átomo {no1+1:>3} ({dados1['simbolo']:>2}) — grau {G.degree(no1)}"

            if i < len(coluna2):
                no2, dados2 = coluna2[i]
                texto2 = f"Átomo {no2+1:>3} ({dados2['simbolo']:>2}) — grau {G.degree(no2)}"
                print(f"{texto1:<35}{texto2}")
            else:
                print(texto1)
    else:
        for no, dados in nos:
            grau = G.degree(no)
            print(f"    Átomo {no+1:>3} ({dados['simbolo']:>2}) — grau {grau}")
 
    elementos = {}
    for _, dados in G.nodes(data=True):
        s = dados["simbolo"]
        elementos[s] = elementos.get(s, 0) + 1
 
    print(f"\n  Composição:")
    for el, qtd in sorted(elementos.items()):
        print(f"    {el}: {qtd}")
 
    print(f"{'='*50}\n")

### Visualizando a molécula em 3D

A última etapa do pipeline é a visualização. A função abaixo recebe o grafo e constrói o gráfico 3D interativo usando o Plotly, montando-o em camadas — chamadas de *traces* na terminologia da biblioteca.

A primeira camada desenha as **ligações** como linhas cinzas no espaço 3D. Para isso, as coordenadas dos dois átomos de cada ligação são adicionadas a listas separadas por eixo (x, y, z), intercaladas com `None` — recurso que instrui o Plotly a interromper a linha entre uma ligação e outra, evitando que todas fiquem conectadas continuamente.

A segunda camada desenha os **átomos** como esferas, agrupados por elemento para que cada um apareça com sua cor e tamanho correspondentes na legenda. Ao passar o mouse sobre um átomo, o gráfico exibe seu símbolo e índice.

Por fim, o layout define a aparência geral: fundo escuro, eixos ocultos e legenda com os elementos presentes, resultando em uma visualização limpa e focada na geometria da molécula.

In [22]:
def plotar_molecula_3d(G, titulo="Molécula 3D"):
    
    traces = []
 

    x_lig, y_lig, z_lig = [], [], []
 
    for i, j in G.edges():
        xi, yi, zi = G.nodes[i]["x"], G.nodes[i]["y"], G.nodes[i]["z"]
        xj, yj, zj = G.nodes[j]["x"], G.nodes[j]["y"], G.nodes[j]["z"]
 
        x_lig += [xi, xj, None]
        y_lig += [yi, yj, None]
        z_lig += [zi, zj, None]
 
    trace_ligacoes = go.Scatter3d(
        x=x_lig, y=y_lig, z=z_lig,
        mode="lines",
        line=dict(color="#888888", width=4),
        hoverinfo="none",        # sem tooltip nas ligações
        name="Ligações",
    )
    traces.append(trace_ligacoes)
 
    grupos = {}
    for no, dados in G.nodes(data=True):
        s = dados["simbolo"]
        if s not in grupos:
            grupos[s] = {"x": [], "y": [], "z": [], "texto": []}
        grupos[s]["x"].append(dados["x"])
        grupos[s]["y"].append(dados["y"])
        grupos[s]["z"].append(dados["z"])
        grupos[s]["texto"].append(f"{s} (átomo {no+1})")
 
    for simbolo, coords in grupos.items():
        cor   = cores_atomos.get(simbolo, cor_padrao)
        raio  = raio_atomos.get(simbolo, raio_padrao)
 
        trace_atomos = go.Scatter3d(
            x=coords["x"], y=coords["y"], z=coords["z"],
            mode="markers+text",
            marker=dict(
                size=raio,
                color=cor,
                line=dict(color="black", width=1),
                opacity=0.9,
            ),
            text=simbolo,
            textposition="top center",
            hovertext=coords["texto"],
            hoverinfo="text",
            name=simbolo,
        )
        traces.append(trace_atomos)
 

    layout = go.Layout(
        title=dict(text=titulo, font=dict(size=20)),
        scene=dict(
            xaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False),
            zaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False),
            bgcolor="black",
        ),
        paper_bgcolor="#111111",
        font=dict(color="white"),
        legend=dict(title="Elementos"),
        margin=dict(l=0, r=0, t=50, b=0),
    )
 
    fig = go.Figure(data=traces, layout=layout)
    fig.show()

### Juntando tudo

Com todas as funções definidas, a função abaixo representa o pipeline completo do projeto. Ela recebe o SMILES da molécula e, opcionalmente, um nome para exibição — caso o nome não seja fornecido, ele é buscado automaticamente no PubChem. Em seguida, executa as quatro etapas em sequência: constrói a molécula em 3D, converte em grafo, imprime a análise e exibe a visualização interativa.

In [ ]:
def visualizar_molecula(smiles: str, nome: str = None):
    
    if nome is None:
        nome = smiles_para_nome(smiles)
 
    # Passo 1 — gera a molécula com coordenadas 3D
    mol = construir_molecula(smiles)
    if mol is None:
        return 
 
    # Passo 2 — converte em grafo NetworkX
    G = molecula_para_grafo(mol)
 
    # Passo 3 — imprime análise do grafo
    analisar_grafo(G, nome_molecula=nome)
 
    # Passo 4 — plota em 3D
    plotar_molecula_3d(G, titulo=f"{nome}  |  SMILE: {smiles}")


## Experimente você mesmo

Agora que todo o pipeline está construído, é sua vez de explorar! Digite o SMILES de qualquer molécula no campo abaixo e o programa irá gerar automaticamente a análise do grafo e a visualização 3D interativa.

Caso não saiba o SMILES da molécula que deseja explorar, você pode consultá-lo no site do [PubChem](https://pubchem.ncbi.nlm.nih.gov/) — basta pesquisar pelo nome da molécula e o SMILES estará disponível na página.

In [27]:
print("Bem vindo(a) ao visuamolécula!!")
print()
print("Aqui você consegue entrar com um smile e visualizar sua molécula e algumas de suas informações.")
print()
print()
print("Tente você, alguns exemplos de smiles: C | O | N | CC | S | COO | P | F")
print()
molecula_usuario = input("Digite o smile da mólecula que deseja visualizar:")

visualizar_molecula(molecula_usuario)


Bem vindo(a) ao visuamolécula!!

Aqui você consegue entrar com um smile e visualizar sua molécula e algumas de suas informações.


Tente você, alguns exemplos de smiles: C | O | N | CC | S | COO | P | F


  Análise da molécula — trans-2-Butene
  Nós (átomos)       : 12
  Arestas (ligações) : 11
  Diâmetro do grafo  : 5

  Grau de cada átomo (nº de ligações):
    Átomo   1 ( C) — grau 4
    Átomo   2 ( C) — grau 3
    Átomo   3 ( C) — grau 3
    Átomo   4 ( C) — grau 4
    Átomo   5 ( H) — grau 1
    Átomo   6 ( H) — grau 1
    Átomo   7 ( H) — grau 1
    Átomo   8 ( H) — grau 1
    Átomo   9 ( H) — grau 1
    Átomo  10 ( H) — grau 1
    Átomo  11 ( H) — grau 1
    Átomo  12 ( H) — grau 1

  Composição:
    C: 4
    H: 8



## Referências

WEININGER, D. SMILES, a chemical language and information system. 1. Introduction to methodology and encoding rules. **Journal of Chemical Information and Computer Sciences**, v. 28, n. 1, p. 31–36, 1988. DOI: 10.1021/ci00057a005.

LANDRUM, G. et al. **RDKit: Open-Source Cheminformatics**. Versão 2026.03.2. Disponível em: https://www.rdkit.org. Acesso em: 18 jun. 2026.

HAGBERG, A. A.; SCHULT, D. A.; SWART, P. J. Exploring network structure, dynamics, and function using NetworkX. In: **Proceedings of the 7th Python in Science Conference (SciPy 2008)**, Pasadena, CA, EUA, ago. 2008. p. 11–15.

PLOTLY TECHNOLOGIES INC. **Collaborative data science**. Montreal, QC: Plotly Technologies Inc., 2015. Disponível em: https://plotly.com. Acesso em: 18 jun. 2026.

KIM, S. et al. PubChem 2023 update. **Nucleic Acids Research**, v. 51, n. D1, p. D1373–D1380, 2023. DOI: 10.1093/nar/gkac956.